# Player Number-Guided AI Highlight Video Curator
**CSCI 5922 — Mateusz Muszynski & Colin Wallace**

Run this notebook on **Google Colab** with a free T4 GPU for full-speed inference.

### Before you start
1. `Runtime → Change runtime type → T4 GPU`
2. Upload your match video to **Google Drive** (any folder, e.g. `My Drive/soccer/`)
3. Run cells top-to-bottom — **do not skip any cell**

---
## 0 — Check GPU

In [ ]:
import torch
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU, then re-run")

---
## 1 — Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/csci5922-highlight-curator.git"  # ← update this
REPO_DIR = "/content/csci5922"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install all dependencies (~2 min first run, cached after)
!pip install -q \
    ultralytics>=8.2.0 \
    supervision>=0.21.0 \
    opencv-python-headless \
    albumentations \
    SoccerNet \
    ffmpeg-python \
    imageio[ffmpeg] \
    PyYAML tqdm rich

import torch, torchvision, ultralytics, supervision, cv2
print(f"torch {torch.__version__} | torchvision {torchvision.__version__}")
print(f"ultralytics {ultralytics.__version__} | supervision {supervision.__version__}")
print(f"opencv {cv2.__version__}")

---
## 2 — Mount Google Drive & set video path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─── EDIT THESE TWO LINES ─────────────────────────────────────────────────────
VIDEO_PATH    = "/content/drive/MyDrive/soccer/Fairview High School vs Boulder High School ｜ Round of 16 ｜ CHSAA 2025 Boy Soccer ｜ #FAIRVIEWBOULDER.mp4"
TARGET_JERSEY = 10   # jersey number you want highlights for
# ──────────────────────────────────────────────────────────────────────────────

OUTPUT_PATH = f"/content/drive/MyDrive/soccer/highlights_jersey{TARGET_JERSEY}.mp4"

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}\nCheck the path above."
print(f"✓ Video found: {os.path.getsize(VIDEO_PATH)/1e9:.2f} GB")

---
## 3 — Trim to a 10-minute game clip
Skips the pre-game intro. Edit `START_SEC` if you want a different window.
Remove this cell (and set `VIDEO_PATH` back) to process the full 2-hour match.

In [ ]:
START_SEC    = 960   # 16:00 — past pre-game intro, into live play
DURATION_SEC = 600   # 10 minutes
CLIP_PATH    = "/content/game_clip.mp4"

!ffmpeg -y -ss {START_SEC} -i "{VIDEO_PATH}" -t {DURATION_SEC} \
    -c copy "{CLIP_PATH}" -loglevel error

print(f"Clip: {os.path.getsize(CLIP_PATH)/1e6:.1f} MB")
VIDEO_PATH = CLIP_PATH  # pipeline uses the clip from here on

---
## 4 — Build model weights
Model weights are **not stored in git** (too large). This cell trains the jersey CNN
and LSTM scorer on synthetic stub data in ~5 min on a T4.  
It is **skipped automatically** if weights already exist (e.g. on a second run).

In [ ]:
import os
os.chdir(REPO_DIR)

jersey_weights = os.path.join(REPO_DIR, "models/jersey_cnn.pt")
scorer_weights = os.path.join(REPO_DIR, "models/scorer_lstm.pt")

needs_training = not os.path.exists(jersey_weights) or not os.path.exists(scorer_weights)

if needs_training:
    print("Weights not found — running quick-train on synthetic stubs (~5 min on T4)...")
    !python scripts/download_soccernet.py --task stubs
    !python run_training.py --quick-test --stages jersey scorer --device cuda
    print("\n✓ Training complete")
else:
    print("✓ Weights already exist — skipping training")
    print(f"  {jersey_weights}")
    print(f"  {scorer_weights}")

# Final guard — raise immediately if weights still missing after training
assert os.path.exists(jersey_weights), f"jersey_cnn.pt still missing after training — check logs above"
assert os.path.exists(scorer_weights), f"scorer_lstm.pt still missing after training — check logs above"
print("\n✓ Both weight files confirmed present")

---
## 5 — Run the full pipeline

In [ ]:
import os, sys
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Re-import in case kernel state changed
from importlib import invalidate_caches
invalidate_caches()
from main import run_pipeline

DEBUG_VIDEO = "/content/debug_overlay.mp4"

result = run_pipeline(
    video_path=VIDEO_PATH,
    jersey_number=TARGET_JERSEY,
    output_path=OUTPUT_PATH,
    config_path=os.path.join(REPO_DIR, "config.yaml"),
    device="cuda",
    conf_override=0.25,
    highlight_thresh_override=0.40,
    frame_skip=1,            # T4 handles every frame at ~80 fps
    clip_length_override=90, # 3-second LSTM windows
    clip_stride_override=15,
    debug_video=DEBUG_VIDEO,
)
print(f"\nHighlight reel → {result}")

---
## 6 — Preview debug overlay
Green box = jersey #{TARGET_JERSEY}. All other tracked players shown in colour.

In [ ]:
PREVIEW = "/content/debug_preview.mp4"
!ffmpeg -y -i {DEBUG_VIDEO} -vcodec libx264 -acodec aac \
    -vf "scale=960:-2" -crf 28 -preset fast {PREVIEW} -loglevel error

from IPython.display import HTML, display
from base64 import b64encode
data = open(PREVIEW, 'rb').read()
b64  = b64encode(data).decode()
display(HTML(f'''
<h3>Debug overlay — jersey #{TARGET_JERSEY} highlighted in green</h3>
<video width="960" controls>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
'''))

---
## 7 — Preview highlight reel

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

if result and os.path.exists(result):
    REEL_PREVIEW = "/content/reel_preview.mp4"
    !ffmpeg -y -i "{result}" -vcodec libx264 -acodec aac \
        -crf 26 -preset fast {REEL_PREVIEW} -loglevel error
    data = open(REEL_PREVIEW, 'rb').read()
    b64  = b64encode(data).decode()
    display(HTML(f'''
    <h3>Highlight reel — jersey #{TARGET_JERSEY}</h3>
    <video width="960" controls>
      <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    '''))
else:
    print("No highlights assembled.")
    print("→ Try lowering highlight_thresh_override to 0.30 in Cell 5 and re-run.")

---
## 8 — Jersey frequency scan
Samples 300 frames and shows which jersey numbers the model actually reads.
Use this to pick a good `TARGET_JERSEY` value for Cell 2.

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from src.detector import PlayerDetector
from src.jersey_reader import JerseyReader

det = PlayerDetector(conf_threshold=0.25, device="cuda")
jr  = JerseyReader(
    model_path=os.path.join(REPO_DIR, "models/jersey_cnn.pt"),
    device="cuda"
)

cap     = cv2.VideoCapture(VIDEO_PATH)
total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
counter = Counter()
indices = np.linspace(0, total - 1, min(300, total // 10), dtype=int)

for fi in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
    ret, frame = cap.read()
    if not ret: continue
    for d in det.detect(frame):
        crop = det.crop_upper_body(frame, d)
        num, conf = jr.read(crop)
        if num is not None:
            counter[num] += 1
cap.release()

df = pd.DataFrame(counter.most_common(20), columns=["jersey", "sightings"])
print(df.to_string(index=False))

plt.figure(figsize=(10, 4))
plt.bar(df["jersey"].astype(str), df["sightings"], color="steelblue")
plt.xlabel("Jersey number")
plt.ylabel("Sightings")
plt.title("Jersey number frequency across sampled frames")
plt.tight_layout()
plt.savefig("/content/jersey_frequency.png", dpi=150)
plt.show()
print("\n→ Set TARGET_JERSEY in Cell 2 to any number above, then re-run Cells 5–7.")